# T04 · The theory behind T05: Hessian supervision of MACE-OFF23_medium — what E/F leaves free, the Cartesian Hessian loss, its estimator, and what frequencies read from it

**openQHA theory notebook (rewritten 2026-09-21 to match T05 and ruling S0-C-53).** T05 states
the goal — fine-tune MACE-OFF23_medium with PHL's loss so that its energies, forces **and
Hessians** reach ωB97M-D3(BJ)/def2-TZVPPD — and gives the recipe. This notebook is the
derivation underneath it. Every statement is derived from the Taylor expansion and the
invariances of the energy, and every identity is checked in the cell that follows it on

* `tests/data/propanal_molecule/`: the ωB97M-D3(BJ)/def2-TZVPPD analytic Hessian and the stored
  MACE-OFF23_medium Hessian at the same basin geometry (T03's pair);
* `tests/data/methyloxirane_frames/`: the Frame set of 2-methyloxirane (QM9 000044) — the basin
  and its four classical-298 K displaced frames, each with **energy, forces and the analytic
  Hessian at both levels at the same positions**: the off-minimum reference Hessians that
  PHL's every-frame rule (T05 §4) makes the campaign buy;
* MACE-OFF23_medium itself, when it loads (the cells say "skipped" otherwise).

**The ruling this notebook serves (S0-C-53).** The training target is the Cartesian Hessian
matrix as stored, $\|H_\theta-H_r\|_F^2$, sampled by Hessian–vector products — exactly the
literature's objective (Rodriguez, PHL, HIP, PFT). Frequencies, normal modes and the msRRHO
entropy are *evaluation quantities* computed from the trained matrix afterwards; they are
not terms of the loss. The projected, mass-weighted norms of T03 stay in the code as
diagnostics. Consequently this notebook derives three things the earlier version did not
need: what the Cartesian norm *contains* (§2, §3), how the frequency and mode errors are
bounded by and read from it (§3), and the estimator in exactly PHL's normalisation (§4).

Reading order: §1 what E/F training leaves free (the curvature at and between samples, and
what a displaced force does constrain) → §2 the exact identities every Hessian field obeys,
and what the rigid block of the Cartesian error is → §3 the Cartesian loss, the other norms
as one operator, and the frequency error read from the Cartesian error (perturbation theory
and Weyl's bound) → §4 the estimator: unbiased for any metric, exact with $3N$ unit probes →
§5 the surface off the minimum: third derivatives, the off-minimum error probe $\rho_k$,
cubic constants → §6 the Hessian–vector product, its gradient and its cost, measured → §7 the
total objective with every frame labelled and replay frames unlabelled → §8 complete
algorithm listings → §9 acceptance, the code map, and how T05 uses each piece.

Conventions: $N$ atoms, $x\in\mathbb R^{3N}$ (Å), $E$ in eV, $g=\nabla E=-F$, $H=\nabla^2E$
(eV Å$^{-2}$), masses $M$ (amu), $K=M^{-1/2}HM^{-1/2}$; $\hat R=\mathrm{blockdiag}(R,\dots,R)$ for
a $3\times3$ rotation $R$; $\|\cdot\|_F$ Frobenius, $\|\cdot\|_2$ spectral; $\theta$ the model
parameters, subscript $r$ the reference.

In [1]:
import sys, math, time
from pathlib import Path
import numpy as np
import torch
torch.set_default_dtype(torch.float64)

def _repo_root():
    for p in [Path.cwd()] + list(Path.cwd().parents):
        if (p / "openqha" / "__init__.py").is_file():
            return p
    raise RuntimeError("run from inside the openQHA checkout")

ROOT = _repo_root(); sys.path.insert(0, str(ROOT))
from ase.data import atomic_masses, atomic_numbers
from openqha.qm_interfaces import orca
from openqha.thermochem import hessian as hessian_mod, hessian_compare as hc, thermo
from openqha.data import frames as frames_mod
from openqha.training import phl, hvp as hvp_mod

LEVEL = "wb97m-d3bj_def2-tzvppd"; MLEVEL = "mace-off23_medium"
to_cm = lambda lam: np.sign(lam) * np.sqrt(np.abs(lam)) * hessian_mod.CM_INV_PER_SQRT_EV_A2_AMU
masses_of = lambda atoms: np.array([atomic_masses[atomic_numbers[s]] for s in atoms.get_chemical_symbols()])

# --- propanal, basin 0: reference H_r and MACE H_t at the wB97M minimum (T03's data)
FIXP = ROOT / "tests" / "data" / "propanal_molecule"
parsed = orca.parse_hess(FIXP / "msrrho" / f"orca.{LEVEL}.basin00.hess")
sym_p = list(parsed["symbols"])
xp = np.asarray(parsed["positions_bohr"]) / orca.BOHR_PER_ANGSTROM
mp = np.array([atomic_masses[atomic_numbers[s]] for s in sym_p])
Hp_r = orca.hessian_to_ev_per_angstrom2(parsed["hessian_eh_bohr2"])
Hp_t = np.load(FIXP / "mace" / "basin00" / f"hessian_at_{LEVEL}.npy")
Np = len(sym_p)

# --- 2-methyloxirane: basin + 4 displaced frames, E/F/H at both levels, identical positions
FIXM = ROOT / "tests" / "data" / "methyloxirane_frames"
fr_t = frames_mod.read_frames(FIXM / f"basin.{MLEVEL}.extxyz") + frames_mod.read_frames(FIXM / f"displaced.{MLEVEL}.extxyz")
fr_r = frames_mod.read_frames(FIXM / f"basin.{LEVEL}.extxyz") + frames_mod.read_frames(FIXM / f"displaced.{LEVEL}.extxyz")
assert all(np.abs(a.positions - b.positions).max() < 1e-7 for a, b in zip(fr_t, fr_r))
Nm = len(fr_t[0]); n3m = 3 * Nm
mm = masses_of(fr_t[0])
X = [a.positions.copy() for a in fr_r]                                   # x_0 (basin), x_1..x_4 (displaced)
E_r = [a.get_potential_energy() for a in fr_r]; E_t = [a.get_potential_energy() for a in fr_t]
G_r = [-a.get_forces().reshape(-1) for a in fr_r]; G_t = [-a.get_forces().reshape(-1) for a in fr_t]   # gradients g = -F
H_r = [np.asarray(a.info["hessian"], float).reshape(n3m, n3m) for a in fr_r]
H_t = [np.asarray(a.info["hessian"], float).reshape(n3m, n3m) for a in fr_t]
rms = [a.info["rms_displacement_A"] for a in fr_t]
print(f"propanal: N = {Np}; 2-methyloxirane ({fr_t[0].info['smiles']}): N = {Nm}, {len(fr_t)} frames, RMS displacements {np.round(rms, 3)} Å")
print("frame   E_r-E_0 (kcal/mol)   |g_r|max (eV/Å)   |g_t|max   |H_r-H_r^T|max   |H_t-H_t^T|max")
for k in range(len(fr_r)):
    print(f"  {k}      {(E_r[k]-E_r[0])*23.0605:8.2f}            {np.abs(G_r[k]).max():6.3f}         {np.abs(G_t[k]).max():6.3f}       {np.abs(H_r[k]-H_r[k].T).max():.1e}          {np.abs(H_t[k]-H_t[k].T).max():.1e}")

propanal: N = 10; 2-methyloxirane (C[C@H]1CO1): N = 10, 5 frames, RMS displacements [0.    0.058 0.096 0.127 0.059] Å
frame   E_r-E_0 (kcal/mol)   |g_r|max (eV/Å)   |g_t|max   |H_r-H_r^T|max   |H_t-H_t^T|max
  0          0.00             0.067          0.000       0.0e+00          0.0e+00
  1          7.25             1.734          1.705       0.0e+00          0.0e+00
  2          4.33             1.039          1.054       0.0e+00          0.0e+00
  3          8.87             2.077          2.130       0.0e+00          0.0e+00
  4          4.90             2.039          1.981       0.0e+00          0.0e+00


## 1. What energy–force training leaves free: the curvature at and between samples

**Proposition 1 (the free function).** Let $\{x_n\}$ be any finite set of training geometries
with pairwise distance $\ge 2\rho>0$, and $\{S_n\}$ any symmetric matrices. There is a smooth
$\varphi$ with
$$\varphi(x_n)=0,\qquad \nabla\varphi(x_n)=0,\qquad \nabla^2\varphi(x_n)=S_n\quad\text{for every }n .$$
*Proof.* Take a bump $b\in C^\infty(\mathbb R)$ with $b(t)=1$ for $|t|\le\tfrac12$, $b(t)=0$ for
$|t|\ge1$, and set
$$\varphi(x)=\sum_n \tfrac12\,(x-x_n)^{\!\top}S_n(x-x_n)\;b\!\left(\frac{\|x-x_n\|}{\rho}\right).$$
The supports are disjoint, so near $x_n$ only term $n$ is present and there $b\equiv1$: $\varphi$
is the quadratic $\tfrac12 d^{\!\top}S_nd$, $d=x-x_n$, whose value and gradient vanish at $d=0$
and whose Hessian is $S_n$. $\square$

So $\tilde E=E+\varphi$ has **the same energies and the same forces as $E$ at every training
geometry** and an arbitrary Hessian there. An E/F loss cannot tell $E$ from $\tilde E$; whatever
curvature the fitted model has at (and between) the samples is decided by the model class and
the optimiser, not by the data. That is the mechanism behind two facts T05 rests on: MACE-OFF23
was trained on 951,813 E/F frames and its Hessians were never constrained (T05 §1 measures
what that left: 0.07–0.08 eV Å$^{-2}$ per element, 5–6 cm$^{-1}$ per mode); and PFT's observation
that E/F fine-tuning on displaced frames improved forces and *degraded* phonons. It is the
reason a Hessian term is in the loss at all, and — since the proposition holds at every sample,
stationary or not — the reason PHL labels every frame with $H$ rather than only the minima.

**What a displaced force does constrain.** The forces at two points are not independent of
the Hessian between them. By the fundamental theorem of calculus along the segment
$x_0+t\,d$, $t\in[0,1]$:
$$g(x_0+d)-g(x_0)=\int_0^1 H(x_0+t\,d)\,d\;dt\;=\;\bar H_d\,d ,\qquad
\bar H_d:=\int_0^1H(x_0+td)\,dt .\tag{1.1}$$
A pair (basin, displaced frame) with forces at both therefore fixes **one column-direction of
the path-averaged Hessian**, $\bar H_d\,d$: $3N$ numbers per frame, out of the $\sim\tfrac12(3N)^2$
of $H$. With the classical 298 K draw the displacement is a sum over all modes, so (1.1)
mixes every mode's curvature into one vector; four frames per basin give four such vectors.
The trapezoid $\bar H_d\approx\tfrac12(H(x_0)+H(x_0+d))$ is exact to $O(\|d\|^3)$ (Simpson's
error term with the third derivative). The cell measures both on the reference labels: how
far the basin Hessian alone is from the force difference (the size of the change of curvature
along $d$) and how far the trapezoid is (the third-derivative remainder). The last column is
the relative change of the reference Hessian itself between the basin and the frame.

In [2]:
# Proposition 1 in one dimension, numerically: E(x) = cos x on samples x_n; E~ = E + phi with S_n = +2, -3, +5
b = lambda t: np.where(np.abs(t) < 1, np.exp(1 - 1 / np.maximum(1 - t ** 2, 1e-300)), 0.0)   # smooth bump, b(0)=1
xs = np.array([0.5, 2.0, 3.5]); S = np.array([2.0, -3.0, 5.0]); rho = 0.6
E = lambda x: np.cos(x)
phi = lambda x: sum(0.5 * s * (x - xn) ** 2 * b((x - xn) / rho) for xn, s in zip(xs, S))
h = 1e-4
d1 = lambda f, x: (f(x + h) - f(x - h)) / (2 * h); d2 = lambda f, x: (f(x + h) - 2 * f(x) + f(x - h)) / h ** 2
print("at the samples:  E~-E   F~-F        H~-H     S_n")
for xn, s in zip(xs, S):
    print(f"   x={xn:3.1f}      {phi(xn):.1e}  {d1(phi, xn):.1e}   {d2(phi, xn):+.6f}  {s:+.1f}")

# (1.1) on the reference labels of 2-methyloxirane: g(x_k) - g(x_0) vs the trapezoid and vs H(x_0) d alone
print("\n(1.1) reference level, per displaced frame k:  |dg|   |dg - H0 d| / |dg|   |dg - ½(H0+Hk) d| / |dg|   ‖H_r(x_k)-H_r(x_0)‖/‖H_r(x_0)‖")
for k in range(1, 5):
    d = (X[k] - X[0]).reshape(-1); dg = G_r[k] - G_r[0]
    e0 = np.linalg.norm(dg - H_r[0] @ d) / np.linalg.norm(dg)
    e1 = np.linalg.norm(dg - 0.5 * (H_r[0] + H_r[k]) @ d) / np.linalg.norm(dg)
    dH = np.linalg.norm(H_r[k] - H_r[0]) / np.linalg.norm(H_r[0])
    print(f"   k={k}  RMS {rms[k]:.3f} Å:   {np.linalg.norm(dg):6.3f}      {e0:.3f}                {e1:.4f}                    {dH:.3f}")
print("-> at 0.06-0.13 Å RMS (a classical 298 K frame) the reference Hessian has changed by 15-22 %, and the basin Hessian alone")
print("   misses the force difference by 20-90 %; the trapezoid (two Hessians) gets it to 1-19 %, the remainder being the third")
print("   derivative along d. A label set with Hessians at the basins only says nothing about this change -- PHL's rule (T05 §4)")
print("   labels every frame, and §5 below measures what that buys.")

at the samples:  E~-E   F~-F        H~-H     S_n
   x=0.5      0.0e+00  0.0e+00   +2.000000  +2.0
   x=2.0      0.0e+00  -3.3e-16   -3.000000  -3.0
   x=3.5      0.0e+00  0.0e+00   +5.000000  +5.0

(1.1) reference level, per displaced frame k:  |dg|   |dg - H0 d| / |dg|   |dg - ½(H0+Hk) d| / |dg|   ‖H_r(x_k)-H_r(x_0)‖/‖H_r(x_0)‖
   k=1  RMS 0.058 Å:    4.018      0.190                0.0135                    0.149
   k=2  RMS 0.096 Å:    2.537      0.905                0.1285                    0.152
   k=3  RMS 0.127 Å:    4.695      0.749                0.1892                    0.223
   k=4  RMS 0.059 Å:    4.129      0.189                0.0145                    0.164
-> at 0.06-0.13 Å RMS (a classical 298 K frame) the reference Hessian has changed by 15-22 %, and the basin Hessian alone
   misses the force difference by 20-90 %; the trapezoid (two Hessians) gets it to 1-19 %, the remainder being the third
   derivative along d. A label set with Hessians at the basins only says n

## 2. Exact identities of the Hessian field — and what the rigid block of the Cartesian error is

Every potential that is a function of interatomic geometry (the reference and MACE alike)
satisfies $E(\hat Rx+\hat{\mathbf 1}c)=E(x)$ for every rotation $R$ and translation $c$. Differentiating
the invariance gives identities that hold **at every geometry**, stationary or not:

**(i) Symmetry.** $H=H^{\!\top}$ (Schwarz).

**(ii) Translation — the acoustic sum rule.** $\nabla E(x+\varepsilon\hat{\mathbf 1}_\alpha)=\nabla E(x)$
for all $\varepsilon$; the $\varepsilon$-derivative at 0 gives $H\hat{\mathbf 1}_\alpha=0$, $\alpha=1,2,3$:
the three translations are in the kernel of $H$ everywhere, $\sum_j H_{i\alpha,j\beta}=0$.

**(iii) Rotation — the gradient identity.** From $E(\hat Rx)=E(x)$, the chain rule gives
$\hat R^{\!\top}\,(\nabla E)(\hat Rx)=\nabla E(x)$, i.e. the gradient is equivariant,
$g(\hat Rx)=\hat R\,g(x)$. Put $R=I+\varepsilon\Omega+O(\varepsilon^2)$ with $\Omega$ antisymmetric
($\Omega u=\omega\times u$) and expand both sides to first order:
$$g(x)+\varepsilon\,H\,\hat\Omega x+O(\varepsilon^2)=g(x)+\varepsilon\,\hat\Omega\,g(x)+O(\varepsilon^2)
\quad\Longrightarrow\quad \boxed{\,H\,\hat\Omega x=\hat\Omega\,g\,}\tag{2.1}$$
for each of the three generators $\Omega_\alpha$. At a stationary point the right side is zero
and the rotations join the kernel; **away from it the Hessian applied to a rigid rotation
is exactly the rotated gradient.** This is the "gradient term" measured on 2026-09-17
(S0-C-43: 29 cm$^{-1}$ on the converged propanal minimum, ~100–180 at displaced frames) — not
noise, and not curvature: (2.1) says the rotational block of $H$ carries *no information beyond
$g$*. (The identity is origin-free: $\hat\Omega(x-c)=\hat\Omega x-\hat\Omega c$ and $\hat\Omega c$ is a
uniform translation, killed by (ii).)

**(iv) Equivariance of the field.** Differentiating $g(\hat Rx)=\hat Rg(x)$ once more:
$H(\hat Rx)=\hat R\,H(x)\,\hat R^{\!\top}$ — the Hessian at a rotated geometry is the rotated Hessian.
MACE's energy is built from invariant features, so (i)–(iv) hold for $H_\theta$ to machine
precision; the reference Hessian obeys them to the accuracy of its integration grid, which is
the cell's second number.

**Corollary (the rigid block of the error is the force error).** In mass-weighted coordinates
the rotation vectors are $r_\alpha=M^{1/2}\hat\Omega_\alpha(x-x_{\rm com})/c_\alpha$ (a normalisation
$c_\alpha$), so by (2.1)
$$K\,r_\alpha=M^{-1/2}HM^{-1/2}\,M^{1/2}\hat\Omega_\alpha(x-x_{\rm com})/c_\alpha
=M^{-1/2}\hat\Omega_\alpha\,g/c_\alpha ,\qquad K\,t_\alpha=0 .\tag{2.2}$$
Orthonormalising $\{t_\alpha,r_\alpha\}$ into $V=[t,r]\,C$ (a $6\times6$ coefficient matrix $C$),
$$KV=\big[\,0,\;M^{-1/2}\hat\Omega_1 g/c_1,\;M^{-1/2}\hat\Omega_2g/c_2,\;M^{-1/2}\hat\Omega_3g/c_3\,\big]\,C ,$$
and for the *error* $\Delta K=K_\theta-K_r$, with $\Delta g=g_\theta-g_r$:
$$\Delta K\,V=\big[\,0,\;M^{-1/2}\hat\Omega_\alpha\,\Delta g/c_\alpha\,\big]\,C ,\qquad
\|\Delta K\,V\|_F\;\le\;\frac{\|C\|_2}{\sqrt{m_{\min}}}\Big(\sum_\alpha\|\hat\Omega_\alpha\Delta g\|^2/c_\alpha^2\Big)^{1/2}
\;\le\;\frac{\sqrt3\,\|C\|_2}{\sqrt{m_{\min}}\,\min_\alpha c_\alpha}\,\|\Delta g\| .\tag{2.3}$$
With $P=I-VV^{\!\top}$ the Frobenius norm splits orthogonally,
$$\|\Delta K\|_F^2=\|P\Delta KP\|_F^2+2\,\|P\Delta K\,V\|_F^2+\|V^{\!\top}\Delta K\,V\|_F^2 ,\tag{2.4}$$
and, for a potential that obeys (2.1) exactly, the last two terms are functions of $\Delta g$
alone by (2.3). MACE obeys (2.1) to machine precision; a DFT Hessian obeys it only to the
accuracy of its integration grid, so the reference's rigid block is $M^{-1/2}\hat\Omega g_r/c$
*plus a residual that is grid noise* — the cell measures it at 2–5 % of the rotated gradient
on the displaced frames, consistent with the "97–98 % explained" of ticket 03.

**What this says about the Cartesian target.** The same split holds for $\Delta H=M^{1/2}\Delta KM^{1/2}$
with the Cartesian rigid vectors. So the Cartesian loss $\|\Delta H\|_F^2$ that S0-C-53 trains on
*contains* the force error a second time (bounded by (2.3)) and the label's invariance
residual, and the rest — the projected part — is the curvature. This is not an argument
against the Cartesian target; it is its content, stated so the judge knows where to look: the
double-counted force error is small where the forces are already good (0.1 % of $\|\Delta K\|^2$,
0.03 % of $\|\Delta H\|^2$ on these frames — the cell's last two columns), it grows with $\|\Delta g\|$ at foreign geometries
(~100 cm$^{-1}$ in S0-C-43/44), and the projected diagnostics of T03 measure the curvature part
alone whenever that separation is wanted.

In [3]:
def omega_hat(alpha, n):
    '''The generator Ω̂_α as a 3N x 3N block matrix: (Ω̂ u)_i = e_α × u_i (the cross-product matrix of e_α, per atom).'''
    e = np.eye(3)[alpha]
    O = np.array([[0.0, -e[2], e[1]], [e[2], 0.0, -e[0]], [-e[1], e[0], 0.0]])
    return np.kron(np.eye(n), O)

def rotation_identity(H, g, x):
    '''(2.1): the absolute residual max_α |H Ω̂x − Ω̂g| (eV/Å) and the size max_α |Ω̂g| it is measured against;
    (ii): the translation sum rule |H 1_α| / |H|.'''
    n = len(x); xc = (x - x.mean(0)).reshape(-1)
    res, size = [], []
    for a in range(3):
        Om = omega_hat(a, n)
        lhs, rhs = H @ (Om @ xc), Om @ g
        res.append(np.linalg.norm(lhs - rhs)); size.append(np.linalg.norm(rhs))
    t = np.zeros((3 * n, 3)); t[0::3, 0] = t[1::3, 1] = t[2::3, 2] = 1.0
    return max(res), max(size), np.linalg.norm(H @ t) / np.linalg.norm(H)

print("(2.1) |H Ω̂x − Ω̂g| (eV/Å) against |Ω̂g|, and (ii) |H 1|/|H|, per frame:")
print("frame   |g|     reference: |H Ω̂x−Ω̂g|   |Ω̂g|    |H1|/|H|      MACE: |H Ω̂x−Ω̂g|   |Ω̂g|    |H1|/|H|")
for k in range(5):
    rr, sr, tr = rotation_identity(H_r[k], G_r[k], X[k]); rt, st, tt = rotation_identity(H_t[k], G_t[k], X[k])
    print(f"  {k}    {np.linalg.norm(G_r[k]):6.3f}       {rr:.1e}       {sr:.3f}   {tr:.1e}            {rt:.1e}       {st:.3f}   {tt:.1e}")
print("-> MACE obeys both identities to machine precision (an invariant energy, differentiated). The reference obeys the sum rule")
print("   to 1e-4 and the rotation identity to 2-5 % of |Ω̂g| at the displaced frames: the DFT grid breaks rotational invariance,")
print("   and at the basin, where |Ω̂g| is ~0.03 eV/Å, the same ~0.02 eV/Å residual is the 24 cm^-1 'noise floor' of ticket 03.\n")

# (2.4) the orthogonal split of the mass-weighted error at every frame, and (2.2): each level's rigid block vs its rotated gradient
print("(2.4)  ‖ΔK‖²_F = ‖PΔKP‖² + 2‖PΔKV‖² + ‖VᵀΔKV‖²      (2.2)  ‖K V − M^-½ Ω̂ g C/c‖_F / ‖K V‖_F per level")
print("frame   ‖ΔK‖²      ‖PΔKP‖²    2‖PΔKV‖²   ‖VᵀΔKV‖²   sum/total    MACE      reference   ‖ΔK V‖_F  ‖M^-½Ω̂Δg C/c‖_F   rigid share of ‖ΔH‖²")
for k in range(5):
    m3 = np.repeat(mm, 3); Minv = 1 / np.sqrt(m3)
    dK = (H_t[k] - H_r[k]) / np.sqrt(np.outer(m3, m3))
    V, sing, rank = hessian_mod.rigid_body_vectors(mm, X[k]); P = np.eye(n3m) - V @ V.T
    a = np.sum((P @ dK @ P) ** 2); bterm = 2 * np.sum((P @ dK @ V) ** 2); c = np.sum((V.T @ dK @ V) ** 2); tot = np.sum(dK ** 2)
    xc = (X[k] - np.average(X[k], axis=0, weights=mm)).reshape(-1)
    raw = np.zeros((n3m, 6))
    for al in range(3):
        raw[al::3, al] = np.sqrt(m3[al::3]); raw[:, 3 + al] = np.sqrt(m3) * (omega_hat(al, Nm) @ xc)
    Cc = np.linalg.lstsq(raw, V, rcond=None)[0]                       # V = raw C
    pred_of = lambda g: np.column_stack([np.zeros((n3m, 3))] + [Minv * (omega_hat(al, Nm) @ g) for al in range(3)]) @ Cc
    K_t = H_t[k] / np.sqrt(np.outer(m3, m3)); K_r = H_r[k] / np.sqrt(np.outer(m3, m3))
    res_t = np.linalg.norm(K_t @ V - pred_of(G_t[k])) / np.linalg.norm(K_t @ V)
    res_r = np.linalg.norm(K_r @ V - pred_of(G_r[k])) / np.linalg.norm(K_r @ V)
    # the same split for the CARTESIAN error with the Cartesian rigid vectors (V_c = M^-1/2 V, re-orthonormalised)
    Vc, _ = np.linalg.qr(Minv[:, None] * V); Pc = np.eye(n3m) - Vc @ Vc.T
    dH = H_t[k] - H_r[k]; rigid_cart = 1 - np.sum((Pc @ dH @ Pc) ** 2) / np.sum(dH ** 2)
    print(f"  {k}   {tot:9.5f}   {a:9.5f}   {bterm:9.5f}   {c:9.5f}    {(a+bterm+c)/tot:.6f}    {res_t:.1e}   {res_r:.3f}      {np.linalg.norm(dK @ V):.4f}    {np.linalg.norm(pred_of(G_t[k] - G_r[k])):.4f}          {rigid_cart*100:5.2f} %")
print("-> the split (2.4) is exact. At the displaced frames MACE's rigid block IS its rotated gradient (2.2) to 1e-7 and the reference's")
print("   is its rotated gradient to 1-2 % (grid; at the basin both blocks are ~0 and the ratio says nothing). So ΔK V = the force")
print("   error (column 9) + the reference's invariance residual, of the same size here. In the Cartesian norm the rigid block is")
print("   the last column: the part of the training target that is the force error counted twice -- 0.03-0.04 % on these frames.")

(2.1) |H Ω̂x − Ω̂g| (eV/Å) against |Ω̂g|, and (ii) |H 1|/|H|, per frame:
frame   |g|     reference: |H Ω̂x−Ω̂g|   |Ω̂g|    |H1|/|H|      MACE: |H Ω̂x−Ω̂g|   |Ω̂g|    |H1|/|H|
  0     0.146       7.6e-02       0.132   9.7e-05            4.7e-07       0.000   1.2e-16
  1     4.081       6.6e-02       3.597   1.3e-04            4.8e-07       3.559   1.3e-16
  2     2.473       9.9e-02       2.140   9.4e-05            4.0e-07       2.127   1.3e-16
  3     4.740       1.2e-01       4.414   7.8e-05            5.4e-07       4.396   1.2e-16
  4     4.053       1.4e-01       3.591   1.1e-04            6.3e-07       3.595   1.1e-16
-> MACE obeys both identities to machine precision (an invariant energy, differentiated). The reference obeys the sum rule
   to 1e-4 and the rotation identity to 2-5 % of |Ω̂g| at the displaced frames: the DFT grid breaks rotational invariance,
   and at the basin, where |Ω̂g| is ~0.03 eV/Å, the same ~0.02 eV/Å residual is the 24 cm^-1 'noise floor' of ticket 03.

(2

## 3. The Cartesian loss, the other norms as one operator, and what frequencies read from it

Everything a second-derivative loss can measure is a norm of the error operator in some
metric. Write, for a fixed linear map $B\in\mathbb R^{m\times3N}$ that depends on the frame
only (masses, positions, the reference label),
$$\mathcal L_B(\theta)=\frac1{\nu}\big\|B\,(H_\theta-H_r)\,B^{\!\top}\big\|_F^2 ,\tag{3.1}$$
with a normalisation $\nu$ ($9N^2$ in PHL's convention for $B=I$, $n_{\rm vib}$ in T03's for the
projected norms). The choices of $B$ in the literature and in this repository:

| name | $B$ | $\nu$ | what it measures | status |
|---|---|---|---|---|
| **Cartesian** | $I$ | $9N^2$ | $\|\Delta H\|_F^2$, eV$^2$ Å$^{-4}$: the matrix as stored | **the training target (S0-C-53)**; Rodriguez 2025, PHL, HIP, PFT |
| flat (mass-weighted, projected) | $PM^{-1/2}$ | $n_{\rm vib}$ | $\|P\Delta KP\|_F^2$: eq. (1) of T03 | diagnostic, `phl.projected_loss_full` |
| weighted in the reference-mode basis | $P_W M^{-1/2}$, $P_W=L_rW^{1/2}L_r^{\!\top}$ | $n_{\rm vib}$ | $\sum_{ij}w_iw_j(D_{ij}-\lambda_i\delta_{ij})^2$: eq. (3) of T03 | diagnostic, `phl.weighted_projector` |
| — entropy weights | $w_i=|\partial S/\partial\omega_i|$ | | the msRRHO entropy's own sensitivity | T03 §3 |
| — relative weights | $w_i=1/|\lambda_i|$ | | relative curvature per mode, i.e. relative frequency error | this section |

**The mode-basis form.** With $L_r$ the reference modes ($L_rL_r^{\!\top}=P$, $L_r^{\!\top}L_r=I$) and
$D=L_r^{\!\top}\tilde K_\theta L_r$, $\Lambda_r=L_r^{\!\top}\tilde K_rL_r$ (T03 eq. 2):
$$\|P_W\Delta KP_W\|_F^2=\|L_rW^{1/2}(D-\Lambda_r)W^{1/2}L_r^{\!\top}\|_F^2
=\|W^{1/2}(D-\Lambda_r)W^{1/2}\|_F^2=\sum_{ij}w_iw_j\,(D_{ij}-\lambda_i\delta_{ij})^2 ,\tag{3.2}$$
using $\|L_rXL_r^{\!\top}\|_F^2=\operatorname{tr}(L_rX^{\!\top}L_r^{\!\top}L_rXL_r^{\!\top})=\operatorname{tr}(X^{\!\top}X)$.

**Relative weights and the frequency.** $\omega_i=\sqrt{\lambda_i}$, so $\delta\omega_i/\omega_i=\tfrac12\,\delta\lambda_i/\lambda_i+O(\delta^2)$.
With $w_i=1/|\lambda_i|$ the diagonal of (3.2) is
$$\sum_i\frac{(D_{ii}-\lambda_i)^2}{\lambda_i^2}=\sum_i\Big(\frac{\delta\lambda_i}{\lambda_i}\Big)^2\approx4\sum_i\Big(\frac{\delta\omega_i}{\omega_i}\Big)^2 :\tag{3.3}$$
the relative loss is four times the mean squared relative frequency error along the
reference modes. (Modes below a floor $\lambda_{\rm floor}$ — imaginary or under 30 cm$^{-1}$ — get
$w=1/\lambda_{\rm floor}$.)

**The Cartesian loss in the mode basis.** $\|\Delta H\|_F^2=\|M^{1/2}\Delta KM^{1/2}\|_F^2=\sum_{ij}m_im_j\Delta K_{ij}^2$,
so
$$m_{\min}^2\|\Delta K\|_F^2\le\|\Delta H\|_F^2\le m_{\max}^2\|\Delta K\|_F^2 ,\tag{3.4}$$
and for the projected part, with $G:=L_r^{\!\top}ML_r$ (the mass metric in the mode basis, *not* diagonal),
$$\|M^{1/2}P\Delta KPM^{1/2}\|_F^2=\|M^{1/2}L_r(D-\Lambda_r)L_r^{\!\top}M^{1/2}\|_F^2=\operatorname{tr}\big[(D-\Lambda_r)\,G\,(D-\Lambda_r)\,G\big] .\tag{3.5}$$
The Cartesian loss is therefore the mode-basis loss with a full (non-diagonal) weight
matrix $G$ that couples modes through the masses: it weights hydrogenic modes by $\sim1$ and
heavy-atom modes by $\sim m_{\rm C}m_{\rm O}\approx190$. That is what the target optimises, and the
cell shows where its gradient goes by frequency band.

**How the frequencies read from the Cartesian error — the post-hoc quantities of S0-C-53.**
The frequencies are the eigenvalues of the *projected mass-weighted* matrix, $\tilde K L=L\Lambda$,
$\omega_i=\sqrt{\lambda_i}$. Two exact statements connect the trained Cartesian error to them:

*(a) First-order perturbation.* For a symmetric perturbation $\Delta\tilde K$ of $\tilde K_r$ with
eigenpairs $(\lambda_i,L_{r,i})$,
$$\delta\lambda_i=L_{r,i}^{\!\top}\,\Delta\tilde K\,L_{r,i}+O(\|\Delta\tilde K\|^2)=D_{ii}-\lambda_i+O(\|\Delta\tilde K\|^2),\qquad
\delta\omega_i=\frac{\delta\lambda_i}{2\omega_i}+O(\delta^2),\tag{3.6}$$
i.e. the diagonal of the mode-basis error *is* the frequency error to first order, and the
off-diagonal $D_{ij}$ is the mode mixing (`hessian_compare`'s `OMEGA_ALONG_REF_CM` and `MIXING`).
The first-order eigenvector change is $\delta L_i=\sum_{j\ne i}\frac{D_{ij}}{\lambda_i-\lambda_j}L_{r,j}$: mixing
is amplified by near-degeneracy, which is why the low-mode row of the judge reads the
*along-reference-mode* curvature $D_{ii}$ and not a sorted spectrum.

*(b) Weyl's inequality — a uniform bound.* For symmetric matrices, the $i$-th sorted eigenvalues
satisfy $|\lambda_i(\tilde K_\theta)-\lambda_i(\tilde K_r)|\le\|\Delta\tilde K\|_2\le\|\Delta\tilde K\|_F$; with (2.4) and (3.4),
$$|\delta\lambda_i|\;\le\;\|P\Delta KP\|_F\;\le\;\|\Delta K\|_F\;\le\;\frac{\|\Delta H\|_F}{m_{\min}}
\qquad\text{for every mode }i ,\tag{3.7}$$
so **driving the Cartesian norm to zero drives every frequency error to zero** — the target
is sufficient for the post-hoc quantities. What (3.7) does *not* say is that the Cartesian
gradient is spent evenly across modes: the chain is loose by orders of magnitude for a soft
mode (a Frobenius norm over $n_{\rm vib}^2$ entries against one eigenvalue, and $m_{\min}=1$ for
hydrogen against heavy-atom curvature — the cell: 1571 cm$^{-1}$ allowed, 4 observed), and the
band table shows the gradient concentrates in the 1000–2000 cm$^{-1}$ band. Whether the low modes follow is therefore a
question for the judge, not a theorem — which is exactly the ruling's division of labour.
The cell verifies (3.2)–(3.7) on the propanal base model.

In [4]:
# propanal, basin: the norms and their band budgets (base model); (3.6)-(3.7) the frequency error read from the Cartesian error
m3p = np.repeat(mp, 3); Kp_r = Hp_r / np.sqrt(np.outer(m3p, m3p)); Kp_t = Hp_t / np.sqrt(np.outer(m3p, m3p))
L_r, lam_r = phl.reference_modes(Hp_r, mp, xp); om_r = to_cm(lam_r); n_vib = L_r.shape[1]
D, _ = phl.mode_basis_terms(Hp_t, Hp_r, mp, xp)
Dm = D - np.diag(lam_r)                                           # the error in the mode basis, D - Λ_r

lam_floor = (30.0 / hessian_mod.CM_INV_PER_SQRT_EV_A2_AMU) ** 2
w_abs = np.ones(n_vib)
w_rel = 1.0 / np.maximum(np.abs(lam_r), lam_floor)
w_ent, _ds = phl.entropy_weights(om_r, mp, xp)
G = L_r.T @ np.diag(m3p) @ L_r

def loss_W(w): return np.sum(np.outer(w, w) * Dm ** 2) / n_vib
print("(3.2) weighted loss via P_W vs mode-basis sum:")
for name, w in (("flat", w_abs), ("relative", w_rel), ("entropy", w_ent)):
    print(f"   {name:9s}  {phl.projected_loss_full(Hp_t, Hp_r, mp, xp, weights=w):.10e}  {loss_W(w):.10e}")
Vp, _s, _r = hessian_mod.rigid_body_vectors(mp, xp); Pp = np.eye(3 * Np) - Vp @ Vp.T
cart_proj = np.sum((np.sqrt(m3p)[:, None] * (Pp @ (Kp_t - Kp_r) @ Pp) * np.sqrt(m3p)[None, :]) ** 2)
print(f"(3.5) ‖M^½ PΔKP M^½‖² = {cart_proj:.10e}   tr[(D-Λ)G(D-Λ)G] = {np.trace(Dm @ G @ Dm @ G):.10e}")
print(f"(3.4) m_min² ‖ΔK‖² ≤ ‖ΔH‖² ≤ m_max² ‖ΔK‖²:  {mp.min()**2*np.sum((Kp_t-Kp_r)**2):.3e} ≤ {np.sum((Hp_t-Hp_r)**2):.3e} ≤ {mp.max()**2*np.sum((Kp_t-Kp_r)**2):.3e}")
om_along = to_cm(np.diag(D)); rel_freq = (om_along - om_r) / om_r
print(f"(3.3) Σ_i (δλ_i/λ_i)² = {np.sum(((np.diag(D) - lam_r) / lam_r) ** 2):.5f}   4 Σ_i (δω_i/ω_i)² = {4 * np.sum(rel_freq ** 2):.5f}   (equal to first order in δ)")

# (3.6) first-order perturbation against the exact eigenvalues of the model's projected matrix
pt = hc.projected(Hp_t, mp, xp); lam_t_sorted = pt["lam"]
d_exact = lam_t_sorted - lam_r                                   # sorted-eigenvalue difference (Weyl's pairing)
d_first = np.diag(D) - lam_r                                     # (3.6) along the reference modes
om_exact = to_cm(lam_t_sorted)
print(f"\n(3.6) frequency error, first order δλ_i = D_ii − λ_i against the exact sorted spectrum (cm⁻¹), propanal base model:")
print("  mode   ω_r      exact δω   first-order δω   |diff|     max_j |D_ij|/|λ_i−λ_j| (mixing amplification)")
for i in list(range(4)) + [n_vib - 2, n_vib - 1]:
    mix = max(abs(Dm[i, j]) / abs(lam_r[i] - lam_r[j]) for j in range(n_vib) if j != i)
    print(f"  {i:3d}   {om_r[i]:7.1f}   {om_exact[i]-om_r[i]:+8.2f}   {to_cm(np.diag(D))[i]-om_r[i]:+8.2f}       {abs((om_exact[i]-om_r[i])-(to_cm(np.diag(D))[i]-om_r[i])):.2f}          {mix:.3f}")
# (3.7) Weyl: every |δλ_i| ≤ ‖PΔKP‖_2 ≤ ‖PΔKP‖_F ≤ ‖ΔK‖_F ≤ ‖ΔH‖_F / m_min
A_proj = Pp @ (Kp_t - Kp_r) @ Pp
print(f"(3.7) max_i |δλ_i| = {np.abs(d_exact).max():.4e}  ≤  ‖PΔKP‖_2 = {np.linalg.norm(A_proj, 2):.4e}  ≤  ‖PΔKP‖_F = {np.linalg.norm(A_proj):.4e}"
      f"  ≤  ‖ΔK‖_F = {np.linalg.norm(Kp_t-Kp_r):.4e}  ≤  ‖ΔH‖_F/m_min = {np.linalg.norm(Hp_t-Hp_r)/mp.min():.4e}   [eV Å⁻² amu⁻¹]")
print(f"      in cm⁻¹ for the softest mode ({om_r[0]:.0f}): the bound allows |δω| ≤ {np.linalg.norm(Hp_t-Hp_r)/mp.min()/(2*np.sqrt(lam_r[0]))*hessian_mod.CM_INV_PER_SQRT_EV_A2_AMU:.0f}, the model's is {abs(om_exact[0]-om_r[0]):.1f}: sufficient, not tight")

# band budgets: share of each norm from pairs (i, j) with max(ω_i, ω_j) in a band
edges = [0, 300, 1000, 2000, 4000]; om_pair = np.maximum.outer(om_r, om_r)
print("\nshare of the loss by band of max(ω_i, ω_j), propanal base model:")
print("norm          " + "".join(f"{lo:>5d}-{hi:<5d}" for lo, hi in zip(edges[:-1], edges[1:])) + "   RMS rel. freq err by band")
for name, w in (("Cartesian *", None), ("flat", w_abs), ("relative", w_rel), ("entropy", w_ent)):
    contrib = (Dm @ G) * (Dm @ G).T if w is None else np.outer(w, w) * Dm ** 2
    tot = contrib.sum(); shares = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = (om_pair >= lo) & (om_pair < hi); shares.append(contrib[sel].sum() / tot)
    bands = [np.sqrt(np.mean(rel_freq[(om_r >= lo) & (om_r < hi)] ** 2)) * 100 if np.any((om_r >= lo) & (om_r < hi)) else np.nan for lo, hi in zip(edges[:-1], edges[1:])]
    print(f"{name:12s}  " + "".join(f"{s:9.1%}  " for s in shares) + "   " + "  ".join(f"{b:5.2f}%" for b in bands))
print("-> * the target. Its gradient is decided above 1000 cm⁻¹ (54 % in 1000-2000, 42 % above); the relative frequency error of the")
print("   base model is largest BELOW 300 (1.69 %). (3.7) guarantees the low modes follow when the Cartesian norm is small enough;")
print("   how fast they follow is the judge's low-mode row, read from the trained matrix by (3.6).")

(3.2) weighted loss via P_W vs mode-basis sum:
   flat       1.2364724403e-02  1.2364724403e-02
   relative   2.6062332774e-03  2.6062332774e-03
   entropy    1.7970472014e-05  1.7970472014e-05
(3.5) ‖M^½ PΔKP M^½‖² = 2.3362820818e+00   tr[(D-Λ)G(D-Λ)G] = 2.3362820818e+00
(3.4) m_min² ‖ΔK‖² ≤ ‖ΔH‖² ≤ m_max² ‖ΔK‖²:  3.017e-01 ≤ 2.332e+00 ≤ 7.600e+01
(3.3) Σ_i (δλ_i/λ_i)² = 0.00570   4 Σ_i (δω_i/ω_i)² = 0.00567   (equal to first order in δ)

(3.6) frequency error, first order δλ_i = D_ii − λ_i against the exact sorted spectrum (cm⁻¹), propanal base model:
  mode   ω_r      exact δω   first-order δω   |diff|     max_j |D_ij|/|λ_i−λ_j| (mixing amplification)
    0     131.1      -4.14      -1.78       2.36          0.139
    1     222.2      +6.51      +5.43       1.08          0.139
    2     256.2      -2.20      -2.18       0.02          0.008
    3     667.3     -12.40     -12.36       0.05          0.033
   22    3144.9      +4.85      +4.86       0.01          0.056
   23    3147.3  

## 4. The estimator: unbiased for any metric, exact with $3N$ unit probes

For $A=B\,\Delta H\,B^{\!\top}$ and a random $v$ with $E[v]=0$, $E[vv^{\!\top}]=I$ (Rademacher or Gaussian),
$$E\|Av\|^2=\operatorname{tr}(A^{\!\top}A\,E[vv^{\!\top}])=\|A\|_F^2 ,\tag{4.1}$$
exactly as in T03 §4 — the proof never used what $B$ is. The model side of $Av=B\,\Delta H\,(B^{\!\top}v)$
is the Hessian–vector product with the **transformed probe** $\tilde v=B^{\!\top}v$:
$$\hat{\mathcal L}_B^{(k)}=\frac1{\nu\,k}\sum_{j=1}^k\big\|B\,(H_\theta\tilde v_j-H_r\tilde v_j)\big\|^2 ,\qquad
\tilde v_j=B^{\!\top}v_j ,\qquad E[\hat{\mathcal L}_B^{(k)}]=\mathcal L_B .\tag{4.2}$$
**For the target, $B=I$: $\tilde v=v$, the probe the model sees is the raw Cartesian draw, the
reference side is the matvec $H_rv$, and $\nu=9N^2$ — (4.2) is then PHL's notebook line by line**
(`hvp = -grad(forces, coords, grad_outputs=v, create_graph=True)`, `true_hvp = bmm(H_ref, v)`,
`/(9N²)`). For the projected diagnostics $\tilde v=M^{-1/2}P_Wv$. The variance (T03 eq. 8) is
$2(\|A^{\!\top}A\|_F^2-\sum_i(A^{\!\top}A)_{ii}^2)/(\nu^2k)$ for Rademacher probes, with any $B$; Gaussian
probes have a larger variance ($\|A\|_F^4$ enters) at the same mean, which is why the code's
default draw is Rademacher.

**Deterministic exactness.** If $U$ is orthonormal with $A=AUU^{\!\top}$ (the columns of $U$ span
$\operatorname{range}(A^{\!\top})$), then $\sum_j\|AU_j\|^2=\operatorname{tr}(A^{\!\top}AUU^{\!\top})=\|A\|_F^2$ with zero variance.
For the Cartesian target $U=I$: **the $3N$ unit probes give $\|\Delta H\|_F^2$ exactly, and they are
the columns of `get_hessian`** (§6 checks this to the last digit) — so the validation loss is
the full matrix, not a sample. For the projected diagnostics the reference modes work,
$U=L_r$ ($n_{\rm vib}$ probes), because $P_WL_rL_r^{\!\top}=P_W$; they are *not* exact for $B=I$,
since $\operatorname{range}(\Delta H)\not\subseteq\operatorname{range}(P)$ (the rigid block of §2). The cell
checks (4.1)–(4.2) for the Cartesian target and for the relative diagnostic on the propanal
base model (4000 draws: the mean is within a few standard errors of the exact value, the
variance matches T03's eq. 8).

In [5]:
rng = np.random.default_rng(0)
Minv_p = 1 / np.sqrt(m3p)
def check_estimator(B, k_draws=4000, gaussian=False):
    A = B @ (Hp_t - Hp_r) @ B.T; fro2 = np.sum(A ** 2)
    draw = (lambda: rng.standard_normal(A.shape[1])) if gaussian else (lambda: rng.choice([-1.0, 1.0], size=A.shape[1]))
    draws = np.array([np.sum((A @ draw()) ** 2) for _ in range(k_draws)])
    Bt = A.T @ A; var_pred = 2 * (np.sum(Bt ** 2) - np.sum(np.diag(Bt) ** 2))
    return fro2, draws.mean(), draws.var(), var_pred
# the target: B = I ; exact set U = I (3N probes)
fro2, mean, var, var_pred = check_estimator(np.eye(3 * Np))
fro2g, meang, varg, _ = check_estimator(np.eye(3 * Np), gaussian=True)
print(f"Cartesian (B = I), PHL's (2.1'):  ‖ΔH‖²_F/(9N²) = {fro2/(9*Np*Np):.6e}")
print(f"   Rademacher mean (4000) = {mean/(9*Np*Np):.6e} ({mean/fro2-1:+.2%}), var {var:.3e} vs (8) {var_pred:.3e};   Gaussian mean = {meang/(9*Np*Np):.6e} ({meang/fro2-1:+.2%}), var {varg:.3e}")
print(f"   exact with U = I ({3*Np} unit probes): {np.sum((Hp_t - Hp_r) ** 2)/(9*Np*Np):.6e};  with U = L_r it is NOT exact: {np.sum(((Hp_t-Hp_r) @ L_r) ** 2)/(9*Np*Np):.6e}")
# the relative diagnostic: B = P_W M^-1/2 ; exact set U = L_r
Bw = phl.weighted_projector(L_r, w_rel) * Minv_p[None, :]
fro2, mean, var, var_pred = check_estimator(Bw)
Aw = Bw @ (Hp_t - Hp_r) @ Bw.T
print(f"relative diagnostic:   ‖A‖²_F = {fro2:.6e}   Rademacher mean (4000) = {mean:.6e} ({mean/fro2-1:+.2%})   var {var:.3e} vs (8) {var_pred:.3e}")
print(f"   exact with U = L_r ({n_vib} probes): {np.sum((Aw @ L_r) ** 2):.6e};  with U = I ({3*Np} probes): {np.sum(Aw ** 2):.6e}")
vt, r, info = phl.make_probes(mp, xp, Hp_r, mode="modes", weights=w_rel)
print(f"phl.make_probes(mode='modes', weights=1/λ): estimator = {phl.estimator_from_products(vt @ Hp_t, r, info):.6e}  = exact {phl.projected_loss_full(Hp_t, Hp_r, mp, xp, weights=w_rel):.6e}")
print("-> (4.1) holds for both metrics; for the target the Rademacher variance is the smaller; the code's B = I path (ticket 17) is the")
print("   first two lines with the probe passed to the model unchanged.")

Cartesian (B = I), PHL's (2.1'):  ‖ΔH‖²_F/(9N²) = 2.590998e-03
   Rademacher mean (4000) = 2.638488e-03 (+1.83%), var 1.407e+00 vs (8) 1.321e+00;   Gaussian mean = 2.597506e-03 (+0.25%), var 2.377e+00
   exact with U = I (30 unit probes): 2.590998e-03;  with U = L_r it is NOT exact: 2.437092e-03
relative diagnostic:   ‖A‖²_F = 6.254960e-02   Rademacher mean (4000) = 6.253643e-02 (-0.02%)   var 2.756e-03 vs (8) 2.767e-03
   exact with U = L_r (24 probes): 6.254960e-02;  with U = I (30 probes): 6.254960e-02
phl.make_probes(mode='modes', weights=1/λ): estimator = 2.606233e-03  = exact 2.606233e-03
-> (4.1) holds for both metrics; for the target the Rademacher variance is the smaller; the code's B = I path (ticket 17) is the
   first two lines with the probe passed to the model unchanged.


## 5. The surface off the minimum: third derivatives, the off-minimum error, cubic constants

**Taylor expansion of the Hessian field.** Along a displacement $d$ from a basin $x_0$,
$$H(x_0+d)=H(x_0)+\mathcal T[d]+O(\|d\|^2),\qquad \mathcal T[d]_{ij}=\sum_k\frac{\partial^3E}{\partial x_i\partial x_j\partial x_k}\,d_k .\tag{5.1}$$
A Hessian label at $x_0$ and a Hessian label at $x_0+d$ together measure one slice
$\mathcal T[d]$ of the third-derivative tensor; a pair of *models* does the same, so the model's
anharmonicity along $d$ can be compared with the reference's:
$$\Delta\mathcal T[d]\;\approx\;\big(H_\theta(x_0+d)-H_\theta(x_0)\big)-\big(H_r(x_0+d)-H_r(x_0)\big) .\tag{5.2}$$

**Why every frame carries a Hessian (PHL's rule, T05 §4).** By Proposition 1 an E/F label at
$x_0+d$ constrains nothing about $H(x_0+d)$ beyond (1.1)'s one direction; by (5.1) the reference
Hessian there differs from the basin's by a whole slice of the cubic tensor. A label set with
Hessians at the minima only would leave $\mathcal T[d]$ to the model class. The measure of what is
at stake is the projected error at a displaced frame against the same error at the basin,
$$\rho_k=\frac{\|P_k\Delta K(x_k)P_k\|_F^2/n_{\rm vib}}{\|P_0\Delta K(x_0)P_0\|_F^2/n_{\rm vib}} ,\tag{5.3}$$
with $P_k$ the projector *at $x_k$* (the rigid block there is the force error, §2, and is removed
on both sides so that $\rho_k$ compares curvature). $\rho_k\gg1$ is the PFT symptom (curvature
roughened between samples). After the fine-tune, $\rho_k$ on held-out displaced frames is a
judge row: it reads whether the displaced Hessian labels were learned or only the basins'.

**Cubic force constants from Hessians along modes.** In mass-weighted normal coordinates
$q=L_r^{\!\top}M^{1/2}(x-x_0)$ the cubic constants are
$$\phi_{ijk}=\frac{\partial^3E}{\partial q_i\partial q_j\partial q_k}
=\frac{\partial}{\partial q_k}\big(L_{r,i}^{\!\top}K(q)L_{r,j}\big)
\approx\frac{D_{ij}(+h\,e_k)-D_{ij}(-h\,e_k)}{2h} ,\qquad D(q)=L_r^{\!\top}K(x_0+M^{-1/2}L_rq)L_r ,\tag{5.4}$$
i.e. **one central difference of two Hessians along mode $k$ gives the whole slice
$\phi_{\cdot\cdot k}$** — $n_{\rm vib}^2$ cubic constants from two Hessians, $O(h^2)$ accurate. The
semi-diagonal constants $\phi_{iik}$ and $\phi_{kkk}$ are what second-order vibrational
perturbation theory (VPT2) reads; for the diagonal anharmonic constant, in wavenumber units
with the Hessian in the same normal coordinates (Barone, *JCP* 122, 014108, 2005),
$$\chi_{ii}=\frac{\phi_{iiii}}{16}-\frac1{16}\sum_k\frac{\phi_{iik}^2\,(8\omega_i^2-3\omega_k^2)}{\omega_k(4\omega_i^2-\omega_k^2)} ,\tag{5.5}$$
so the anharmonic fundamental $\nu_i=\omega_i+2\chi_{ii}+\tfrac12\sum_{j\ne i}\chi_{ij}$ depends on
the Hessian *surface* along every mode. A model whose Cartesian Hessians match the reference
at the basin and at the displaced frames matches $\mathcal T[d]$ along the sampled $d$; this is the
harmonic-plus-first-anharmonic content the every-frame rule buys. The cell applies
(5.2)–(5.3) to the 2-methyloxirane frames at both levels and computes (5.4) for the model
along the three softest propanal modes.

In [6]:
# (5.3) the off-minimum error probe and (5.2) the anharmonicity mismatch, 2-methyloxirane, base model
print("2-methyloxirane, base model vs reference (Hessians at both levels at the same frames):")
print("frame  RMS(Å)  L_flat(x_k) [eV²Å⁻⁴amu⁻²/mode]   ρ_k    ‖ΔH‖²/(9N²) (target)   lowest 3 ω_r at x_k        lowest 3 ω_θ at x_k        |T_r[d]|/|H_r|   |ΔT[d]|/|T_r[d]|")
L0 = phl.projected_loss_full(H_t[0], H_r[0], mm, X[0])
for k in range(5):
    Lk = phl.projected_loss_full(H_t[k], H_r[k], mm, X[k])
    cart = np.linalg.norm(H_t[k] - H_r[k]) ** 2 / (9 * Nm * Nm)
    pr_k = hc.projected(H_r[k], mm, X[k]); pt_k = hc.projected(H_t[k], mm, X[k])
    if k:
        Tr = H_r[k] - H_r[0]; Tt = H_t[k] - H_t[0]
        mis = np.linalg.norm(Tt - Tr) / np.linalg.norm(Tr)
    print(f"  {k}    {rms[k]:.3f}   {Lk:.5f}                      {Lk/L0:5.2f}      {cart:.5f}            {np.round(pr_k['freq'][:3], 0)}   {np.round(pt_k['freq'][:3], 0)}   {'-' if k == 0 else f'{np.linalg.norm(Tr)/np.linalg.norm(H_r[0]):.3f}':>8s}         {'-' if k == 0 else f'{mis:.2f}'}")
print("-> ρ_k (5.3): the base model's projected error at a 0.06-0.13 Å frame is 1.45-2.0x its basin error in the squared norm (1.2-1.4x")
print("   in the norm itself, T05 §4), while the reference Hessian")
print("   itself has changed by 15-22 % and the model reproduces that change to 4-8 % (|ΔT[d]|/|T_r[d]|, eq. 5.2). The Cartesian")
print("   target column is what the fine-tune minimises at each of these frames; ρ_k afterwards is the judge's reading of it.")

# (5.4) cubic constants along the three softest propanal modes, MACE (six Hessians)
try:
    from ase import Atoms
    from openqha.potentials import engine
    calc, ename, prov = engine.calculator(device="cpu")
    atoms0 = Atoms(symbols=sym_p, positions=xp)
    h = 0.15                                                             # amu^1/2 Å along the mass-weighted mode
    print(f"\n(5.4) {ename}: cubic slices φ_·,·,k along the three softest propanal modes, h = {h} amu^½ Å  [eV Å⁻³ amu⁻³⁄²]")
    print("  k   ω_k(cm⁻¹)   φ_kkk       max_i |φ_iik|   (i of that max, ω_i)   check: (D(+h)+D(-h))/2 − D(0) diag max (the O(h²) quartic term)")
    for kmode in range(3):
        dq = (Minv_p * L_r[:, kmode]).reshape(Np, 3)
        Dpm = []
        for s in (+1, -1):
            a = atoms0.copy(); a.positions = xp + s * h * dq
            Hk = np.asarray(calc.get_hessian(a)).reshape(3 * Np, 3 * Np)
            Kk = Hk / np.sqrt(np.outer(m3p, m3p))
            Dpm.append(L_r.T @ (Pp @ Kk @ Pp) @ L_r)                     # D(±h) in the BASIN's mode basis (fixed L_r, P at x_0)
        phi_k = (Dpm[0] - Dpm[1]) / (2 * h)
        i_max = int(np.argmax(np.abs(np.diag(phi_k))))
        quart = np.abs(np.diag((Dpm[0] + Dpm[1]) / 2 - D)).max()
        print(f"  {kmode}   {om_r[kmode]:7.1f}     {phi_k[kmode, kmode]:+.4f}     {np.abs(np.diag(phi_k)).max():.4f}       ({i_max}, {om_r[i_max]:.0f})              {quart:.4f}")
    print("  Modes 0-1 are the aldehyde and methyl torsions: a periodic (cosine) potential has no cubic term at its minimum, and the")
    print("  model's φ_kkk there is ~0 -- the right symmetry; the couplings φ_iik that survive are to C-H stretches (Morse-like).")
    print("  The reference's φ_kkk along these modes = the second difference of ticket 32's along-mode energy line (mode_curvature WIDE profile).")
except Exception as exc:                                                 # noqa: BLE001
    print("MACE not available here -- (5.4) skipped:", str(exc)[:160])

2-methyloxirane, base model vs reference (Hessians at both levels at the same frames):
frame  RMS(Å)  L_flat(x_k) [eV²Å⁻⁴amu⁻²/mode]   ρ_k    ‖ΔH‖²/(9N²) (target)   lowest 3 ω_r at x_k        lowest 3 ω_θ at x_k        |T_r[d]|/|H_r|   |ΔT[d]|/|T_r[d]|
  0    0.000   0.02817                       1.00      0.02869            [206. 368. 410.]   [194. 366. 412.]          -         -


  1    0.058   0.04087                       1.45      0.02298            [294. 415. 476.]   [286. 413. 469.]      0.149         0.08
  2    0.096   0.04278                       1.52      0.01728            [373. 404. 477.]   [374. 408. 481.]      0.152         0.07


  3    0.127   0.04298                       1.53      0.02709            [430. 480. 567.]   [434. 486. 567.]      0.223         0.04
  4    0.059   0.05651                       2.01      0.02648            [302. 360. 411.]   [297. 364. 417.]      0.164         0.07
-> ρ_k (5.3): the base model's projected error at a 0.06-0.13 Å frame is 1.45-2.0x its basin error in the squared norm (1.2-1.4x
   in the norm itself, T05 §4), while the reference Hessian
   itself has changed by 15-22 % and the model reproduces that change to 4-8 % (|ΔT[d]|/|T_r[d]|, eq. 5.2). The Cartesian
   target column is what the fine-tune minimises at each of these frames; ρ_k afterwards is the judge's reading of it.


/home/ubuntu/anaconda3/envs/openqha/lib/python3.11/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


/mnt/c/Users/10704/Documents/01_Free-Energy-alchemical/openQHA-Hessian/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)



(5.4) MACE-OFF23_medium: cubic slices φ_·,·,k along the three softest propanal modes, h = 0.15 amu^½ Å  [eV Å⁻³ amu⁻³⁄²]
  k   ω_k(cm⁻¹)   φ_kkk       max_i |φ_iik|   (i of that max, ω_i)   check: (D(+h)+D(-h))/2 − D(0) diag max (the O(h²) quartic term)


  0     131.1     +0.0022     0.0553       (23, 3147)              0.4058


  1     222.2     +0.0020     0.0276       (22, 3145)              0.5119


  2     256.2     -0.1571     1.0032       (18, 2912)              0.0649
  Modes 0-1 are the aldehyde and methyl torsions: a periodic (cosine) potential has no cubic term at its minimum, and the
  model's φ_kkk there is ~0 -- the right symmetry; the couplings φ_iik that survive are to C-H stretches (Morse-like).
  The reference's φ_kkk along these modes = the second difference of ticket 32's along-mode energy line (mode_curvature WIDE profile).


## 6. The Hessian–vector product: gradient and cost, derived and measured

**The product.** With $F_\theta=-\nabla_xE_\theta$ built with a graph (`create_graph=True`), for a
constant $\tilde v$,
$$H_\theta\tilde v=\nabla_x\big(\nabla_xE_\theta\cdot\tilde v\big)=-\nabla_x\big(F_\theta\cdot\tilde v\big) ,\tag{6.1}$$
one more backward pass over the force graph (T03 eq. 9; `hvp.hvp_from_forces`; PHL's
`torch.autograd.grad(forces, coordinates, grad_outputs=vectors, create_graph=True)` is the same
line). Nothing in MACE's forward changes: in training mode the forces already carry the graph
and `batch.positions` is the leaf they were taken against.

**The gradient of the estimator.** Let $\rho_j=B(H_\theta\tilde v_j)-r_j$ with $r_j=BH_r\tilde v_j$ a
constant. Then
$$\frac{\partial\hat{\mathcal L}_B^{(k)}}{\partial\theta}=\frac2{\nu k}\sum_j\rho_j^{\!\top}B\,\frac{\partial(H_\theta\tilde v_j)}{\partial\theta},
\qquad \frac{\partial(H_\theta\tilde v_j)}{\partial\theta}=-\frac{\partial}{\partial\theta}\nabla_x\big(F_\theta\cdot\tilde v_j\big)
=-\frac{\partial^3E_\theta}{\partial\theta\,\partial x\,\partial x}\tilde v_j ,\tag{6.2}$$
a mixed third derivative that autograd delivers when the HVP itself is built with
`create_graph=True` (the "third-order graph"; PFT's "triple backward"). The E/F terms need
only $\partial^2E/\partial\theta\partial x$.

**Cost model.** Reverse-mode AD of a scalar costs a fixed small multiple of the forward it
differentiates (Baur–Strassen). Take one *energy + forces with graph* pass as the unit $u$
(a forward plus one backward, the graph kept). One HVP is one more backward over that
graph — a graph of about the same size — so each probe costs $\approx1\,u$, and the probes
share the force graph: **the Hessian term of a training step costs $\approx k\,u$ on top of
the $1\,u$ the E/F terms already pay**, plus the parameter backward through it. The full
matrix by `vmap` is the same product for the $3N$ unit vectors, $\approx3N\,u$ (vmap batches the
backward passes but does not shorten them). So the ratio full : $k$-probe is $\approx3N:k$ —
for the campaign's 19-atom molecules and $k=4$, $57:4$ per step — and the full-matrix memory
grows with the $3N$ retained graphs. The cell measures the actual wall time on
MACE-OFF23_medium at $N=10$: $u$, then $k=1,4,8,30$ HVPs on the same force graph, then
`get_hessian`, and checks that the $3N$ unit-probe HVPs *are* `get_hessian` — which is why
the validation loss of §4 (the full matrix) costs $3N\,u$ per frame and no more.

In [7]:
try:
    from ase import Atoms
    from openqha.potentials import engine
    calc, ename, prov = engine.calculator(device="cpu")
    atoms0 = Atoms(symbols=sym_p, positions=xp)
    model = calc.models[0]
    def timed(fn, reps=3):
        ts = []
        for _ in range(reps):
            t0 = time.perf_counter(); out = fn(); ts.append(time.perf_counter() - t0)
        return min(ts), out
    b = calc._atoms_to_batch(atoms0)
    def ef():
        bd = calc._clone_batch(b).to_dict()
        out = model(bd, training=True, compute_force=True, compute_stress=False)
        return out["forces"], bd["positions"]
    t_ef, (F_m, pos) = timed(ef)
    rows = []
    for k in (1, 4, 8, 30):
        probes = torch.as_tensor(rng.choice([-1.0, 1.0], size=(k, Np, 3)), dtype=F_m.dtype)
        t_k, _ = timed(lambda: hvp_mod.hvp_from_forces(F_m, pos, probes, create_graph=False))
        rows.append((k, t_k))
    t_full, Hm = timed(lambda: np.asarray(calc.get_hessian(atoms0)).reshape(3 * Np, 3 * Np))
    print(f"{ename}, N = {Np} (3N = {3*Np}), CPU wall (best of 3):")
    print(f"  energy + forces with graph         {t_ef*1e3:7.1f} ms   (= 1 unit)")
    for k, t_k in rows:
        print(f"  {k:2d} HVP on that graph               {t_k*1e3:7.1f} ms   {t_k/t_ef:5.2f} units   {t_k/t_ef/k:5.2f} per probe")
    print(f"  get_hessian (vmap over 3N = {3*Np})    {t_full*1e3:7.1f} ms   {t_full/t_ef:5.2f} units")
    hv30 = hvp_mod.hvp_from_forces(F_m, pos, torch.eye(3 * Np, dtype=F_m.dtype).reshape(3 * Np, Np, 3)).detach().numpy().reshape(3 * Np, 3 * Np)
    print(f"  3N unit-probe HVPs == get_hessian to {np.abs(hv30 - Hm).max():.1e} eV/Å²;  stored fixture Hessian to {np.abs(Hm - Hp_t).max():.1e}")
    # (2.1') on the live model: the Cartesian estimator with k = 4 raw Rademacher probes against the exact ‖ΔH‖²/(9N²)
    probes4 = torch.as_tensor(rng.choice([-1.0, 1.0], size=(4, Np, 3)), dtype=F_m.dtype)
    hv4 = hvp_mod.hvp_from_forces(F_m, pos, probes4, create_graph=False).detach().numpy().reshape(4, -1)
    ref4 = (probes4.numpy().reshape(4, -1)) @ Hp_r.T
    est = np.mean(np.sum((hv4 - ref4) ** 2, axis=1)) / (9 * Np * Np)
    print(f"  (2.1') k = 4 Cartesian probes on the live model: {est:.4e}  against the exact ‖ΔH‖²/(9N²) = {np.sum((Hm - Hp_r) ** 2)/(9*Np*Np):.4e}  (one draw; unbiased over draws, §4)")
    # the third-order graph (6.2)
    params = list(model.parameters())
    for q in params: q.requires_grad_(True)
    try:
        F_g, pos_g = ef()
        hv = hvp_mod.hvp_from_forces(F_g, pos_g, probes[:1], create_graph=True)
        gpar = torch.autograd.grad(hv.pow(2).sum(), params, allow_unused=True)
        nz = sum(g is not None and float(g.abs().max()) > 0 for g in gpar)
        print(f"  (6.2) the HVP carries a graph to θ: {nz} of {len(params)} parameter tensors receive a non-zero gradient")
    finally:
        for q in params: q.requires_grad_(False)
except Exception as exc:                                                 # noqa: BLE001
    print("MACE not available here -- timing skipped:", str(exc)[:160])

MACE-OFF23_medium, N = 10 (3N = 30), CPU wall (best of 3):
  energy + forces with graph            39.8 ms   (= 1 unit)
   1 HVP on that graph                  49.5 ms    1.24 units    1.24 per probe
   4 HVP on that graph                 200.0 ms    5.02 units    1.25 per probe
   8 HVP on that graph                 397.3 ms    9.97 units    1.25 per probe
  30 HVP on that graph                1441.6 ms   36.18 units    1.21 per probe
  get_hessian (vmap over 3N = 30)     1652.7 ms   41.48 units


  3N unit-probe HVPs == get_hessian to 0.0e+00 eV/Å²;  stored fixture Hessian to 7.1e-15
  (2.1') k = 4 Cartesian probes on the live model: 2.2273e-03  against the exact ‖ΔH‖²/(9N²) = 2.5910e-03  (one draw; unbiased over draws, §4)


  (6.2) the HVP carries a graph to θ: 29 of 29 parameter tensors receive a non-zero gradient


## 7. The total objective with every frame labelled — and replay frames unlabelled

Under PHL's rule every frame of the fine-tune Dataset carries $E$, $F$ and $H$; the replay
head's SPICE frames (T05 §6) carry $E$ and $F$ only. Let $\mathcal B$ be a batch of graphs and
$\mathcal B_H\subseteq\mathcal B$ those with a Label. The objective is
$$\mathcal L(\theta)=w_E\,\mathcal L_E(\mathcal B)+w_F\,\mathcal L_F(\mathcal B)+w_H\,\frac1{|\mathcal B_H|}\sum_{n\in\mathcal B_H}\hat{\mathcal L}_{I,n}^{(k)}(\theta) ,\tag{7.1}$$
with mace's own per-graph E/F terms (each frame's `config_weight` multiplying its E/F
terms — the replay weight knob), and the Hessian term averaged over the labelled graphs only.
Four consequences follow from what was derived above:

1. *Masking is exact.* By (4.1) each $\hat{\mathcal L}_{I,n}$ is an unbiased estimate of $\|\Delta H_n\|_F^2/(9N_n^2)$, so
   the masked mean is an unbiased estimate of the masked population loss; graphs without a
   Label (the replay frames) contribute $0$ and no gradient through the Hessian term — they
   still drive E and F, which is their job.
2. *Batching is free.* A batch is a block-diagonal graph, so $-\nabla_x\sum_nF_n\!\cdot\!\tilde v_{n,j}$ returns
   every graph's own $H_n\tilde v_{n,j}$ in one backward (T03 §8; PHL's "single HVP call" with the
   concatenated $v$): the Hessian term costs $k$ backward passes per batch regardless of $|\mathcal B|$;
   an unlabelled graph gets a zero probe.
3. *The weight $w_H$ has no universal value.* $\mathcal L_I$ is in eV$^2$ Å$^{-4}$ and its size depends on
   the molecules and on the base model's state. PHL's $w_H/w_F=0.09/0.30$ was tuned for ANI in
   Hartree from scratch and would put the Hessian term at $10\times$ the force term on our
   fixture (T05 §5). The rule that does transfer: at epoch 0, $w_H\hat{\mathcal L}_H=w_F\mathcal L_F$ on the
   base model, then scan below it.
4. *Two stages.* When mace switches to Stage Two the loss is rebuilt from the `swa_*` weights;
   $w_H^{\rm II}=w_H\,w_F^{\rm II}/w_F$ keeps the Hessian share unchanged (T05 §6).

The cell checks 1–2 on a torch toy potential with a three-graph batch (two labelled, one
not), and the gradient (6.2) by finite differences through the masked batch and the
third-order graph — with $B=I$ for one labelled graph (the target's path) and the projected
$B$ for the other, to show the mask and the gradient are metric-independent.

In [8]:
class ToyPotential(torch.nn.Module):
    '''E(x) = Σ_{i<j} f_θ(r_ij) + Σ_i g_θ(|x_i - x_com|²): pair MLP + a confining term, parameters θ.'''
    def __init__(self):
        super().__init__()
        self.pair = torch.nn.Sequential(torch.nn.Linear(1, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1))
        self.conf = torch.nn.Parameter(torch.tensor(0.05))
    def forward(self, x, batch=None):
        n = x.shape[0]
        idx = torch.zeros(n, dtype=torch.long) if batch is None else batch
        same = (idx[:, None] == idx[None, :]).triu(1)
        diff = x[:, None, :] - x[None, :, :]
        d = torch.sqrt((diff ** 2).sum(-1)[same] + 1e-12).unsqueeze(-1)
        e_pair = self.pair(d).sum()
        com = x.mean(0, keepdim=True) if batch is None else torch.stack([x[idx == g].mean(0) for g in idx.unique()])[idx]
        return e_pair + self.conf * ((x - com) ** 2).sum()

torch.manual_seed(0); toy = ToyPotential()
k = 3
def cartesian_probes(H_ref, n_atoms, k, rng):
    '''B = I (the target): raw Rademacher probes, r_j = H_r v_j, denominator 9 N² k.'''
    v = rng.choice([-1.0, 1.0], size=(k, 3 * n_atoms))
    return v, v @ H_ref.T, dict(projector=np.eye(3 * n_atoms), inv_sqrt_m=np.ones(3 * n_atoms), denominator=9 * n_atoms * n_atoms * k)
# three graphs: propanal (labelled, B = I -- the target), 2-methyloxirane basin (labelled, projected flat diagnostic), displaced k=2 (E/F only, "replay-like")
graphs = [(xp, mp, Hp_r, "cartesian"), (X[0], mm, H_r[0], "flat"), (X[2], mm, None, None)]
def hessian_term(model, graphs, probes_by_graph):
    '''(7.1)'s Hessian term on a block-diagonal batch: one forward, one force backward, k HVP backwards.'''
    xs = torch.tensor(np.concatenate([g[0] for g in graphs]), requires_grad=True)
    idx = torch.cat([torch.full((len(g[0]),), i, dtype=torch.long) for i, g in enumerate(graphs)])
    E = model(xs, idx); F = -torch.autograd.grad(E, xs, create_graph=True)[0]
    offs = np.cumsum([0] + [len(g[0]) for g in graphs])
    total, n_lab = 0.0, 0
    hv_all = []
    for j in range(k):
        probe = torch.zeros_like(xs)
        for n, g in enumerate(graphs):
            if g[2] is not None:
                probe[offs[n]:offs[n+1]] = torch.tensor(probes_by_graph[n][0][j].reshape(-1, 3))
        hv_all.append(torch.autograd.grad(-(F * probe).sum(), xs, create_graph=True)[0])
    for n, g in enumerate(graphs):
        if g[2] is None:
            continue
        vt, r, info = probes_by_graph[n]
        Bt = torch.tensor(info["projector"] * info["inv_sqrt_m"][None, :])      # B rows: I for the target, P_W M^-1/2 for a diagnostic
        acc = 0.0
        for j in range(k):
            hv = hv_all[j][offs[n]:offs[n+1]].reshape(-1)
            rho = Bt @ hv - torch.tensor(r[j])
            acc = acc + rho.pow(2).sum()
        total = total + acc / info["denominator"]; n_lab += 1
    return total / n_lab
probes_by_graph = {0: cartesian_probes(Hp_r, Np, k, np.random.default_rng(10)),
                   1: phl.make_probes(mm, X[0], H_r[0], mode="rademacher", k=k, rng=np.random.default_rng(11))}
L_batch = hessian_term(toy, graphs, probes_by_graph)
L_sep = [hessian_term(toy, [g], {0: probes_by_graph[n]}) for n, g in enumerate(graphs) if g[2] is not None]
print(f"(7.1) masked batch of 3 (2 labelled, one B = I, one projected): {L_batch.item():.12e}   mean of the two alone: {np.mean([l.item() for l in L_sep]):.12e}   diff {abs(L_batch.item() - np.mean([l.item() for l in L_sep])):.1e}")
p = toy.pair[0].weight; g_auto = torch.autograd.grad(L_batch, p)[0][3, 0].item()
eps = 1e-4
with torch.no_grad(): p[3, 0] += eps
Lp = hessian_term(toy, graphs, probes_by_graph).item()
with torch.no_grad(): p[3, 0] -= 2 * eps
Lm = hessian_term(toy, graphs, probes_by_graph).item()
with torch.no_grad(): p[3, 0] += eps
print(f"(6.2) ∂L/∂θ autograd {g_auto:+.8e}   central FD {(Lp - Lm) / (2 * eps):+.8e}   |diff| {abs(g_auto - (Lp - Lm) / (2 * eps)):.1e}")

(7.1) masked batch of 3 (2 labelled, one B = I, one projected): 1.722226896992e+02   mean of the two alone: 1.722226896992e+02   diff 0.0e+00
(6.2) ∂L/∂θ autograd +3.47551776e-01   central FD +3.47551783e-01   |diff| 7.2e-09


## 8. Algorithm listings — complete

Notation as above; `←` assignment; every step names the function that implements it
(`openqha/training/phl.py`, `hvp.py`, `phl_loss.py`, `judge.py`) or the ticket that will
(ticket 17: the $B=I$ path as default).

```
Algorithm 1  frame_constants(x, M, H_r, metric)                  -- once per labelled frame, no autograd
  Input   x [3N] positions, M masses, H_r [3N,3N] raw Cartesian Label (never pre-projected),
          metric ∈ {cartesian (default, S0-C-53), flat, relative, entropy}
  1  if metric = cartesian:  B ← I ;  ν ← 9 N²                          # the target; nothing else is computed
  2  else:
       V ← rigid_body_vectors(M, x);  P ← I − V Vᵀ                        # (phl.projector)
       K_r ← M^-1/2 H_r M^-1/2 ;  K~_r ← P K_r P
       (λ_r, L_r) ← eigh(K~_r) restricted to range(P), ascending             # (phl.reference_modes)
       w ← 1 | 1/max(|λ_i|, λ_floor) | |∂S_msRRHO/∂ω_i|(T)/max            # flat | relative | entropy (phl.entropy_weights)
       B ← L_r W^1/2 L_rᵀ M^-1/2 ;  ν ← n_vib                                # (phl.weighted_projector)
  Output  (B, ν)                                                          # cached by the Label's bytes (phl_loss.FrameConstants)
```

```
Algorithm 2  probes(frame constants, mode, k, rng)                  -- per frame per batch, no autograd (phl.make_probes)
  1  if mode = rademacher:  v_j ~ Uniform{−1,+1}^{3N}, j = 1..k          denominator ← ν · k
     if mode = gaussian:    v_j ~ N(0, I),               j = 1..k          denominator ← ν · k
     if mode = cartesian:   v_j ← e_j,                   j = 1..3N        denominator ← ν       (exact for B = I: = get_hessian, §4/§6)
     if mode = modes:       v_j ← L_r,j,                 j = 1..n_vib     denominator ← ν       (exact for the projected metrics only)
  2  ṽ_j ← Bᵀ v_j                               # the vector the model sees: v itself for the target, M^-1/2 P_W v for a diagnostic
  3  r_j ← B H_r ṽ_j                            # the reference side: a matvec on the Label (PHL's bmm(H_ref, v))
  Output  {ṽ_j}, {r_j}, denominator
```

```
Algorithm 3  training_step(batch 𝓑, θ, k, w_E, w_F, w_H)          -- one optimiser step (phl_loss.forward inside mace's train loop)
  1  for n in 𝓑 with a Label:  ({ṽ_{n,j}}, {r_{n,j}}, den_n) ← Algorithm 2      # replay frames and any unlabelled graph: ṽ_{n,j} ← 0
  2  E_θ, F_θ ← model(𝓑, training=True)          # forces with create_graph=True; batch.positions is the leaf
  3  for j = 1..k:
  4      g_j ← autograd.grad( −Σ_n F_{θ,n} · ṽ_{n,j},  positions,  create_graph=True )   # = [H_{θ,n} ṽ_{n,j}]_n, one backward for the whole batch (6.1)  (hvp.hvp_from_forces)
  5  for n in 𝓑_H:  L̂_n ← Σ_j ‖ B_n g_{n,j} − r_{n,j} ‖² / den_n                 # (4.2), per-graph slice by ptr
  6  L_H ← (1/|𝓑_H|) Σ_{n ∈ 𝓑_H} L̂_n                                               # masked mean (7.1); 0 · Σ F if 𝓑_H = ∅
  7  L ← w_E L_E(𝓑) + w_F L_F(𝓑) + w_H L_H                                       # mace's E/F terms, each frame × its config_weight
  8  θ ← optimizer.step( ∂L/∂θ )                 # the parameter backward runs through the third-order graph of step 4 (6.2)
  (multihead: the replay head's frames are in the same batch and pay E/F only; Stage Two rebuilds this loss from the swa_* weights)
```

```
Algorithm 4  evaluate(validation set, θ)                            -- exact, no probes (phl_loss: wants_hessian_at_eval)
  1  for each labelled frame:  H_θ ← model(frame, compute_hessian=True)      # the full matrix by vmap, create_graph=False (= 3N unit probes, §6)
  2      L_n ← ‖ H_θ − H_r ‖²_F / (9 N²)                                       # the target, exactly
  3  report mean L_n; and the E/F metrics as mace does
  (the logged validation Hessian loss is the training target itself, never a sample)
```

```
Algorithm 5  judge(held-out frames, θ, base)                         -- the ruler (ticket 14's judge; the H rows re-ordered for S0-C-53)
  1  for each held-out frame with a Label, for engine ∈ {θ, base}:
  2      E, F, H ← the engine's own get_potential_energy / get_forces / get_hessian
  3      E rows: per-atom error;  F rows: RMSE                                      (the same level as the base: no zero shift)
  4      H rows, primary:  HESSIAN_MAE, HESSIAN_RMSE, ‖ΔH‖_F/‖H_r‖_F                (the target's own metric)
  5      H rows, post hoc: hessian_compare.compare_hessians(H, H_r, M, x):
             identities (i)-(iii) of §2; D_ii vs λ_r and MIXING (3.6); frequency MAE all modes; low-mode MAE (< 300); softening slope; cos v1
  6      projected diagnostics: L_flat, L_relative, L_entropy by (3.1)              (what the target did to each norm)
  7  for each basin with held-out displaced frames:  ρ_k (5.3), |ΔT[d]|/|T_r[d]| (5.2)
  8  for the pinned seven: thermochemistry at the engine's own minima (msRRHO Records, never recomputed here)
  9  forgetting: E/F RMSE on the fixed SPICE test draw, ratio to base
  10 rows per structure class, per distribution (interpolation / out_of_molecule / in_distribution); thresholds per row;
     calibration: base vs base must not fail a no-degradation line; the 0.9x-scaled engine must fail the low-mode line
```

```
Algorithm 6  cubic_slice(x_0, L_r, M, k, h, engine)                -- (5.4); the anharmonic probe of one mode
  1  δx ← M^-1/2 L_r,k · h
  2  H_± ← engine.get_hessian(x_0 ± δx);  K_± ← M^-1/2 H_± M^-1/2
  3  D_± ← L_rᵀ P K_± P L_r                                      # in the BASIN's mode basis and projector
  4  φ_{·,·,k} ← (D_+ − D_−) / (2h);   quartic check ← (D_+ + D_−)/2 − D_0
  Output  the slice φ_ijk for all i, j; feeds (5.5) with the ω_i
```

## 9. Acceptance, the code map, and how T05 uses each piece

| id | statement | cell | result here |
|---|---|---|---|
| S1 | E/F training cannot fix the curvature at or between samples (Prop. 1) | §1 | $\tilde E$ built with $E,F$ equal, $H$ arbitrary |
| S2 | a displaced force fixes one column-direction of the path-averaged Hessian (1.1) | §1 | trapezoid to 1–19 %; basin $H$ alone off by 20–90 %; $H_r$ changes 15–22 % over a 298 K frame |
| S3 | $H\hat{\mathbf 1}=0$ and $H\hat\Omega x=\hat\Omega g$ at every frame (2.1) | §2 | MACE 1e-7 relative; reference: sum rule 1e-4, rotation 2–5 % (grid) |
| S4 | the rigid block of the error is the force error plus the label's invariance residual, never curvature (2.2–2.4); its share of the Cartesian target | §2 | split exact; MACE block $=$ rotated gradient to 1e-7; 0.1–0.3 % of $\|\Delta H\|^2$ |
| S5 | the norms are one operator with different $B$; (3.2), (3.5) | §3 | to 1e-10 |
| S6 | relative loss $=4\times$ mean squared relative frequency error (3.3); first-order $\delta\omega_i$ from $D_{ii}$ (3.6) | §3 | equal to first order; per-mode table |
| S7 | Weyl: every $|\delta\lambda_i|\le\|\Delta H\|_F/m_{\min}$ (3.7) — the Cartesian target is sufficient for every frequency | §3 | chain of inequalities holds; loose by ~400× on the softest mode |
| S8 | the estimator is unbiased for $B=I$ and for every $P_W$; $3N$ unit probes are exact for the target, $L_r$ is not (4.1–4.2) | §4 | within a few s.e.; exact; variance $=$ (8) |
| S9 | the off-minimum probe $\rho_k$ and $\Delta\mathcal T[d]$ are measurable from the frame fixture (5.2–5.3); cubic slices from two Hessians (5.4) | §5 | $\rho_k$ 1.45–2.0 (squared norm); six MACE Hessians |
| S10 | one HVP $\approx1\,u$; $3N$ HVPs $=$ `get_hessian`; (2.1') on the live model | §6 | 1.0–1.2 $u$ per probe; identical to 0 |
| S11 | masking is exact with mixed metrics and the gradient runs through the third-order graph (6.2, 7.1) | §7 | batch $=$ mean of labelled; FD to 1e-8 |

**Code map.** Algorithm 1: `phl.projector`, `phl.reference_modes`, `phl.entropy_weights`,
`phl.weighted_projector`; the `cartesian` branch ($B=I$, $\nu=9N^2$) is ticket 17 and becomes
the default. Algorithm 2: `phl.make_probes`, `phl.estimator_from_products`. Algorithm 3:
`hvp.hvp_from_forces`, `phl_loss.WeightedEnergyForcesHessianLoss` (the `nn.Module` mace's
`train()` calls through the fork's external-loss hook; commit C keeps it in multihead mode).
Algorithm 4: `pred["hessian"]` at validation (`wants_hessian_at_eval`). Algorithm 5:
`judge.frame_rows` / `aggregate` / `verdict` with `hessian_compare.compare_hessians`.
Algorithm 6: `hvp.along_mode_curvature` gives $D_{kk}$ without the matrix; the slice needs
`get_hessian` at $x_0\pm\delta x$.

**How T05 uses each section.** T05 §1 (the base model's E/F/H numbers) is what S1 predicts
must be there. T05 §2 (PHL's loss is the target, the projected variant a diagnostic) is
§2–§4 here: the rigid block's content (S4), the unbiased estimator in PHL's normalisation
(S8). T05 §3 (where the Cartesian gradient goes) is (3.5) and the band table; T05's judge
rows for frequencies are (3.6)–(3.7): computable from the trained matrix, bounded by the
target, not guaranteed to be tight. T05 §4 (every frame carries $H$) is Proposition 1 plus
(5.1)–(5.3). T05 §5 ($w_H$ measured) is §7 item 3. T05 §6 (replay frames unlabelled, two
stages) is (7.1) and items 1, 4. T05 §8's recipe is Algorithms 1–5 with their flags.

References: Baur–Strassen (reverse-mode cost) — Baur & Strassen, *Theor. Comput. Sci.* 22, 317 (1983);
Weyl, *Math. Ann.* 71, 441 (1912) (eigenvalue perturbation bound); Barone, *J. Chem. Phys.* 122, 014108 (2005) (VPT2, eq. 5.5);
Hutchinson, *Commun. Stat. Simul.* 18, 1059 (1989); PHL (reference implementation `PHL-main/PHL_training.ipynb`);
PFT arXiv:2601.07742; Rodriguez et al. *JCTC* 2025; HIP — `docs/cite/cite_openQHA.bib`; T03 for the
projected loss and its verified identities; T05 for the goal, the recipe and the comparison with the
earlier version of this notebook.